# Show file & Column

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# YOUR 2010 JOB DATA FOLDER
# ============================================================

BASE_DIR = (
    Path.home()
    / "Downloads"
    / "Internship_SCIPE CI-SIP"
    / "MainProject"
    / "2_Job"
    / "JobRawData"
    / "2010_May 2010"
)

print("BASE_DIR:")
print(BASE_DIR)
print()

# ============================================================
# CHECK FOLDER EXISTS
# ============================================================

if not BASE_DIR.exists():
    print("ERROR: Folder does not exist.")
else:
    print("Folder exists.")
    print()

# ============================================================
# SHOW ALL FILES AND FOLDERS
# ============================================================

all_items = sorted(BASE_DIR.rglob("*"))

print("ALL FILES AND FOLDERS:")
for item in all_items:
    if item.is_dir():
        print("[FOLDER]", item.relative_to(BASE_DIR))
    else:
        print("[FILE]  ", item.relative_to(BASE_DIR))

print()
print("Total items:", len(all_items))

# ============================================================
# MAKE A TABLE OF EXCEL FILES ONLY
# ============================================================

excel_files = sorted([
    f for f in BASE_DIR.rglob("*")
    if f.suffix.lower() in [".xls", ".xlsx"]
])

file_table = pd.DataFrame({
    "file_number": range(1, len(excel_files) + 1),
    "folder": [str(f.parent.relative_to(BASE_DIR)) for f in excel_files],
    "file_name": [f.name for f in excel_files],
    "full_path": [str(f) for f in excel_files],
})

print()
print("EXCEL FILE TABLE:")
display(file_table)

# ============================================================
# PREVIEW EACH EXCEL FILE
# ============================================================

for i, file_path in enumerate(excel_files, start=1):
    print("=" * 100)
    print(f"FILE {i}: {file_path.name}")
    print("FOLDER:", file_path.parent.relative_to(BASE_DIR))
    print("FULL PATH:", file_path)
    
    try:
        df = pd.read_excel(file_path, nrows=10)
        
        print("Rows preview:", len(df))
        print("Columns:")
        print(list(df.columns))
        
        display(df.head(10))
        
    except Exception as e:
        print("Could not read this file:")
        print(e)

In [ ]:
from pathlib import Path
import pandas as pd
import time
from openpyxl import Workbook

# ============================================================
# MEMORY-OPTIMIZED COMBINE MAY 2010 JOB RAW EXCEL FILES
#
# INPUT:
# ~/Downloads/Internship_SCIPE CI-SIP/MainProject/2_Job/JobRawData/2010_May 2010
#
# OUTPUT:
# ~/Downloads/all_data_M_2010.xlsx
#
# This version does NOT store all files in memory together.
# It reads one file, writes it, then moves to next file.
# ============================================================

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = (
    Path.home()
    / "Downloads"
    / "Internship_SCIPE CI-SIP"
    / "MainProject"
    / "2_Job"
    / "JobRawData"
    / "2010_May 2010"
)

OUTPUT_FILE = Path.home() / "Downloads" / "all_data_M_2010.xlsx"

YEAR_VALUE = 2010
RELEASE_VALUE = "May 2010"

EXCEL_MAX_ROWS = 1_048_576

# Metadata columns added to every row
META_COLUMNS = [
    "year",
    "release",
    "data_level",
    "source_folder",
    "source_file",
    "source_path",
]

# ============================================================
# CHECK INPUT FOLDER
# ============================================================

print("INPUT FOLDER:")
print(BASE_DIR)
print()

print("OUTPUT FILE:")
print(OUTPUT_FILE)
print()

if not BASE_DIR.exists():
    raise FileNotFoundError(f"Folder does not exist: {BASE_DIR}")

print("Folder exists.")
print()

# ============================================================
# FIND REAL DATA EXCEL FILES
#
# Skip:
# - field_descriptions.xls
# - hidden files
# ============================================================

excel_files = sorted([
    f for f in BASE_DIR.rglob("*.xls")
    if f.name.lower() != "field_descriptions.xls"
    and not f.name.startswith(".")
])

print("REAL DATA EXCEL FILES FOUND:")
for i, f in enumerate(excel_files, start=1):
    print(f"{i}. {f.relative_to(BASE_DIR)}")

print()
print("Total real data files:", len(excel_files))
print()

if len(excel_files) == 0:
    raise ValueError("No real Excel data files found.")

# ============================================================
# DATA LEVEL FUNCTION
# ============================================================

def get_data_level(file_path):
    folder_name = file_path.parent.name.lower()
    file_name = file_path.name.lower()

    if "oesm10nat" in folder_name:
        return "national"

    elif "oesm10st" in folder_name:
        return "state"

    elif "metropolitan" in folder_name:
        return "metro_nonmetro"

    elif "industry" in folder_name or "ownership" in folder_name:
        if "owner" in file_name or "ownership" in file_name:
            return "national_industry_ownership"
        else:
            return "national_industry"

    else:
        return "unknown"

# ============================================================
# FIRST PASS: COLLECT ALL COLUMN NAMES ONLY
#
# This reads headers only, not full data.
# ============================================================

print("=" * 100)
print("FIRST PASS: collecting all column names...")
print()

data_columns = []

for i, file_path in enumerate(excel_files, start=1):
    print(f"Checking columns {i} of {len(excel_files)}: {file_path.name}")

    try:
        header_df = pd.read_excel(
            file_path,
            nrows=0,
            dtype=str,
            engine="xlrd"
        )

        for col in header_df.columns:
            if col not in data_columns:
                data_columns.append(col)

    except Exception as e:
        print("FAILED reading header:")
        print(file_path)
        print(e)

ALL_COLUMNS = META_COLUMNS + data_columns

print()
print("TOTAL FINAL COLUMNS:", len(ALL_COLUMNS))
print()
print("ALL FINAL COLUMNS:")
for col in ALL_COLUMNS:
    print(col)

# ============================================================
# CREATE MEMORY-OPTIMIZED EXCEL WRITER
# ============================================================

wb = Workbook(write_only=True)

sheet_num = 1
current_sheet = None
rows_in_current_sheet = 0

total_rows_written = 0
failed_files = []

def create_new_sheet():
    global sheet_num, current_sheet, rows_in_current_sheet

    sheet_name = f"all_data_{sheet_num}"
    current_sheet = wb.create_sheet(title=sheet_name)

    # Write header row
    current_sheet.append(ALL_COLUMNS)

    rows_in_current_sheet = 1

    print()
    print("=" * 100)
    print(f"Created sheet: {sheet_name}")
    print("=" * 100)

    sheet_num += 1

create_new_sheet()

# ============================================================
# SECOND PASS: READ ONE FILE AT A TIME AND WRITE TO EXCEL
# ============================================================

print()
print("SECOND PASS: reading and writing one file at a time...")
print()

start_time = time.time()

for file_index, file_path in enumerate(excel_files, start=1):
    print("=" * 100)
    print(f"Processing file {file_index} of {len(excel_files)}")
    print(file_path.relative_to(BASE_DIR))

    try:
        # Read only this file into memory
        df = pd.read_excel(
            file_path,
            dtype=str,
            engine="xlrd"
        )

        # Add metadata columns
        df.insert(0, "year", YEAR_VALUE)
        df.insert(1, "release", RELEASE_VALUE)
        df.insert(2, "data_level", get_data_level(file_path))
        df.insert(3, "source_folder", file_path.parent.name)
        df.insert(4, "source_file", file_path.name)
        df.insert(5, "source_path", str(file_path))

        # Make this file match the final master column structure
        df = df.reindex(columns=ALL_COLUMNS)

        # Convert NaN to blank cells
        df = df.astype(object).where(pd.notna(df), None)

        file_rows = 0

        # Write rows one by one
        for row in df.itertuples(index=False, name=None):

            # Excel row limit protection
            if rows_in_current_sheet >= EXCEL_MAX_ROWS:
                create_new_sheet()

            current_sheet.append(row)

            rows_in_current_sheet += 1
            total_rows_written += 1
            file_rows += 1

        print("Rows written from this file:", f"{file_rows:,}")
        print("Total rows written so far:", f"{total_rows_written:,}")

        # Remove this file from memory
        del df

    except Exception as e:
        failed_files.append({
            "file": str(file_path),
            "error": str(e)
        })

        print("FAILED:")
        print(e)

# ============================================================
# OPTIONAL FAILED FILES SHEET
# ============================================================

if failed_files:
    fail_sheet = wb.create_sheet(title="failed_files")
    fail_sheet.append(["file", "error"])

    for item in failed_files:
        fail_sheet.append([item["file"], item["error"]])

# ============================================================
# SAVE OUTPUT FILE
# ============================================================

print()
print("=" * 100)
print("Saving final Excel file...")
print(OUTPUT_FILE)

wb.save(OUTPUT_FILE)

elapsed = time.time() - start_time

print()
print("=" * 100)
print("DONE")
print("Saved here:")
print(OUTPUT_FILE)
print()
print("Total data rows written:", f"{total_rows_written:,}")
print("Total sheets created:", sheet_num - 1)

if failed_files:
    print("Failed files:", len(failed_files))
else:
    print("Failed files: 0")

print(f"Total time: {elapsed:.2f} seconds")